# M1 tensor contract
Build and validate the shared 100 × 60 × F tensor using train-only preprocessing.

In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = _BOOTSTRAP_REPO / 'scripts' / 'colab_bootstrap.py'
if not _BOOTSTRAP_SCRIPT.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/fcb0507351d694c2431e454bd84a357958f96635/scripts/colab_bootstrap.py'
    urllib.request.urlretrieve(raw, '/content/colab_bootstrap.py')
    _BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


In [ ]:
# Build cache and run leakage validation.
import subprocess, sys
ARTIFACTS = WORKSPACE / 'artifacts' / 'v3_tensor_cache'
cmd = [sys.executable, str(REPO / 'scripts' / 'build_v3_tensor_cache.py'), '--data-root', str(DATA_ROOT), '--output-dir', str(ARTIFACTS)]
result = subprocess.run(cmd, check=False, capture_output=True, text=True)
if result.stdout: print(result.stdout)
if result.stderr: print('STDERR:\n' + result.stderr)
if result.returncode != 0: raise RuntimeError(f'tensor cache build failed: {result.stderr or result.stdout}')
leakage_report = WORKSPACE / 'runs' / 'v3_leakage_report.json'
result = subprocess.run([sys.executable, str(REPO / 'scripts' / 'validate_v3_leakage.py'), '--data-root', str(DATA_ROOT), '--report', str(leakage_report)], check=False, capture_output=True, text=True)
if result.stdout: print(result.stdout)
if result.stderr: print('STDERR:\n' + result.stderr)
if result.returncode != 0: raise RuntimeError(f'leakage validation failed: {result.stderr or result.stdout}')
print('Tensor cache ready:', ARTIFACTS)


In [ ]:
# Persist cache and leakage report to Drive.
from shutil import copytree, copy2
drive_runs = Path('/content/drive/MyDrive/kltn/runs')
drive_runs.mkdir(parents=True, exist_ok=True)
copytree(ARTIFACTS, drive_runs / 'artifacts' / 'v3_tensor_cache', dirs_exist_ok=True)
if leakage_report.exists(): copy2(leakage_report, drive_runs / 'v3_leakage_report.json')
print('M1 artifacts synced to:', drive_runs / 'artifacts' / 'v3_tensor_cache')
